# Data Cleaning

## Gathering Data Paths
Data paths were stored in the .env file, and the data itself is excluded from the git repository to avoid any client data leaks.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

data_path = os.getenv("data_path")
accounts_raw_data_path = os.getenv("accounts_data_path")
closure_raw_data_path = os.getenv("closure_data_path")
marketing_raw_data_path = os.getenv("marketing_data_path")

NOTEBOOK_DIR = "../"
ACC_DATA_PATH = NOTEBOOK_DIR + accounts_raw_data_path
CLOSURE_DATA_PATH = NOTEBOOK_DIR + closure_raw_data_path
MARKETING_DATA_PATH = NOTEBOOK_DIR + marketing_raw_data_path

print(f"Accounts data path: {ACC_DATA_PATH}")
print(f"Closure data path: {CLOSURE_DATA_PATH}")
print(f"Marketing data path: {MARKETING_DATA_PATH}")

## Importing Data

In [ ]:
import pandas as pd

In [ ]:

accounts_df = pd.read_csv(ACC_DATA_PATH)
closure_df = pd.read_csv(CLOSURE_DATA_PATH)
marketing_df = pd.read_csv(MARKETING_DATA_PATH)

In [ ]:
print("Accounts DataFrame head:")
print(accounts_df.head())

In [ ]:
print("Accounts DataFrame description:")
print(accounts_df.describe())

In [ ]:
print("Accounts DataFrame info:")
print(accounts_df.info())

In [ ]:
print("Accounts DataFrame null values:")
print(accounts_df.isna().sum())

In [ ]:
print("Accounts DataFrame shape:")
print(accounts_df.shape)

In [ ]:
print("Closure DataFrame head:")
print(closure_df.head())

In [ ]:
print("Closure DataFrame description:")
print(closure_df.describe())

In [ ]:
print("Closure DataFrame info:")
print(closure_df.info())

In [ ]:
print("Closure DataFrame null values:")
print(closure_df.isna().sum())

In [ ]:
print("Closure DataFrame shape:")
print(closure_df.shape)

In [ ]:
print("Marketing DataFrame head:")
print(marketing_df.head())

In [ ]:
print("Marketing DataFrame description:")
print(marketing_df.describe())

In [ ]:
print("Marketing DataFrame info:")
print(marketing_df.info())

In [ ]:
print("Marketing DataFrame null values:")
print(marketing_df.isna().sum())

In [ ]:
print("Marketing DataFrame shape:")
print(marketing_df.shape)

## Data Transformations
A few of the data types could be better adjusted to suit the data stored. As such we'll now perform some transformations to better align our data with our needs.

In [ ]:
'''
Account DataFrame columns before:
 0   customer_id          non-null  int64 
 1   month                non-null  object
 2   acquisition_channel  non-null  object
 3   region               non-null  object
 4   age_band             non-null  object
 5   account_tier         non-null  object
 6   monthly_fee          non-null  int64 
 7   joined_on_promo      non-null  bool  
'''
accounts_df['month'] = pd.to_datetime(accounts_df['month'], format='%Y-%m').dt.to_period('M') # https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.to_period.html
accounts_df['acquisition_channel'] = accounts_df['acquisition_channel'].astype('string')
accounts_df['region'] = accounts_df['region'].astype('string')
age_band_defined_order = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
accounts_df['age_band'] = pd.Categorical(accounts_df['age_band'], categories=age_band_defined_order, ordered=True) # https://pandas.pydata.org/docs/user_guide/categorical.html
account_tier_defined_order = ['Classic', 'Silver', 'Gold', 'Platinum']
accounts_df['account_tier'] = pd.Categorical(accounts_df['account_tier'], categories=account_tier_defined_order, ordered=True)
accounts_df.info()

In [ ]:
'''
Closure DataFrame columns before:
 0   customer_id  non-null   int64 
 1   close_month  non-null   object
 2   reason_code  non-null   object
'''
closure_df['close_month'] = pd.to_datetime(closure_df['close_month'], format='%Y-%m').dt.to_period('M')
closure_df['reason_code'] = closure_df['reason_code'].astype('string')
closure_df.info()

In [ ]:
'''
Marketing DataFrame columns before:
 0   month      non-null    object 
 1   channel    non-null    object 
 2   spend_gbp  non-null    float64
'''
marketing_df['month'] = pd.to_datetime(marketing_df['month'], format='%Y-%m').dt.to_period('M')
marketing_df['channel'] = marketing_df['channel'].astype('string')
marketing_df.info()

## Data Quality Checks
So far we've checked what the data holds and the approximate shapes of the dataframes, in addition to changing some of the data types. Now we should aim to compare values to validate that there are no duplicate ID's, missing values not marked by NaN or Null, or generally anything amiss.

In [ ]:
print("Checking for duplicates in Accounts DataFrame:")
for column_name in accounts_df.columns:
    print(f"{column_name}: {accounts_df[column_name].duplicated().any()}")

In [ ]:
print("Checking for duplicates in Closure DataFrame:")
for column_name in closure_df.columns:
    print(f"{column_name}: {closure_df[column_name].duplicated().any()}")

In [ ]:
print("Checking for duplicates in Marketing DataFrame:")
for column_name in marketing_df.columns:
    print(f"{column_name}: {marketing_df[column_name].duplicated().any()}")

In [ ]:
print("Checking for non-null empty or void values in Accounts DataFrame:")
for column_name in accounts_df.columns:
    non_null_values = accounts_df[column_name].dropna()
    has_empty_or_void = (non_null_values == '').any() or (non_null_values == 0).any()
    print(f"{column_name}: {has_empty_or_void}")

In [ ]:
print("Checking for non-null empty or void values in Closure DataFrame:")
for column_name in closure_df.columns:
    non_null_values = closure_df[column_name].dropna()
    has_empty_or_void = (non_null_values == '').any() or (non_null_values == 0).any()
    print(f"{column_name}: {has_empty_or_void}")

In [ ]:
print("Checking for non-null empty or void values in Marketing DataFrame:")
for column_name in marketing_df.columns:
    non_null_values = marketing_df[column_name].dropna()
    has_empty_or_void = (non_null_values == '').any() or (non_null_values == 0).any()
    print(f"{column_name}: {has_empty_or_void}")

### Analysis of Checks
It seems that all of the three datasets contain duplicates of values in specific individual columns.

We should now identify if any duplicate records exist, or if any duplicates exist across value pairs that should be unique.

In [ ]:
print("Checking for duplicate records in Accounts DataFrame:")
duplicate_accounts = accounts_df[accounts_df.duplicated(keep=False)]
print(f"Number of duplicate records found: {len(duplicate_accounts)}")

In [ ]:
print("Checking for duplicate records in Closure DataFrame:")
duplicate_closure = closure_df[closure_df.duplicated(keep=False)]
print(f"Number of duplicate records found: {len(duplicate_closure)}")

In [ ]:
print("Checking for duplicate records in Marketing DataFrame:")
duplicate_marketing = marketing_df[marketing_df.duplicated(keep=False)]
print(f"Number of duplicate records found: {len(duplicate_marketing)}")

Following these quick checks I can see that there's no entire duplicate records.

We should now check that there are no duplicate records across data value pairs that should be unique.

For a quick reminder of the descriptions given:
 - moneycraft_accounts.csv One row per customer per active month. An account has closed (churned) when its rows stop — there is no churn column.
 - moneycraft_closure_reasons.csv Recorded reasons for closed accounts — but only for some of them. This table is deliberately incomplete: many closures have no reason recorded.
 - moneycraft_marketing_spend.csv One row per month per channel: what we spent winning customers.

In [ ]:
matching_records_accounts = accounts_df[accounts_df.duplicated(subset=["customer_id", "month"], keep=False)].sort_values(by=["customer_id", "month"])
matching_records_accounts # No accounts are duplicated across months.

In [ ]:
matching_records_closure = closure_df[closure_df.duplicated(subset=["customer_id"], keep=False)].sort_values(by=["customer_id"])
matching_records_closure # No closure records are duplicated across customer_id.

In [ ]:
matching_records_marketing = marketing_df[marketing_df.duplicated(subset=["month", "channel"], keep=False)].sort_values(by=["month", "channel"])
matching_records_marketing # No marketing records are duplicated across month and channel   .

## Data Manipulation
We need to add some calculated data fields and generate some new data frames so that we can appropriately find the insights we need.

In [ ]:
'''
Adding Churn and LTV columns to account data frame, before adding closure data for further analysis.
'''

final_month = accounts_df['month'].max()

accounts_grouped = accounts_df.groupby('customer_id').agg(
    signup_month = ('month', 'min'),
    last_active = ('month', 'max'),
    region = ('region', 'last'),
    age_band = ('age_band', 'last'),
    joining_account_tier = ('account_tier', 'first'),
    current_account_tier = ('account_tier', 'last'),
    tenure_months = ('month', 'count'),
    total_revenue = ('monthly_fee', 'sum'),
    acquisition_channel = ('acquisition_channel', 'first'),
    joined_on_promo = ('joined_on_promo', 'first')
)

accounts_grouped['churned'] = accounts_grouped['last_active'] < final_month

accounts_grouped['churn_reason'] = closure_df.groupby('customer_id')['reason_code'].last()

accounts_grouped['LTV'] = accounts_grouped['total_revenue'] / accounts_grouped['tenure_months']
accounts_grouped.head()

In [ ]:
'''
Aggregating account data by acquisition channel.
'''

channels_value = accounts_grouped.groupby('acquisition_channel').agg(
    most_joined_tier = ('joining_account_tier', lambda x: x.value_counts().idxmax() if not x.empty else None), # https://www.reddit.com/r/learnpython/comments/16yw9mz/pandas_how_do_i_get_the_most_frequent_string_in_a/
    most_current_tier = ('current_account_tier', lambda x: x.value_counts().idxmax() if not x.empty else None),
    average_tenure_months = ('tenure_months', 'mean'),
    total_revenue = ('total_revenue', 'sum'),
    most_joined_on_promo = ('joined_on_promo', lambda x: x.value_counts().idxmax() if not x.empty else None),
    most_common_churn_reason = ('churn_reason', lambda x: x.value_counts().idxmax() if not x.empty else None)
)
channels_value.head()

## Summary
We can be sure that the records are now clean enough to work with, and that we can export these values for other processes to work with.

In [ ]:
accounts_df.to_parquet(f'../{data_path}/accounts.parquet')
closure_df.to_parquet(f'../{data_path}/closure.parquet')
marketing_df.to_parquet(f'../{data_path}/marketing.parquet')
accounts_grouped.to_parquet(f'../{data_path}/accounts_summary.parquet')
channels_value.to_parquet(f'../{data_path}/channels_summary.parquet')
# Sources
'''
Google Search AI - used to find the most efficient way to save a pandas dataframe to machine for retrieval in other files.
https://parquet.apache.org/docs/
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html
https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html
'''